In [29]:
!pip install black
!pip install "black[jupyter]"

In [30]:
!black "hirehi_parsing.ipynb"

reformatted hirehi_parsing.ipynb

All done! ✨ 🍰 ✨
1 file reformatted.


In [1]:
import requests

In [2]:
url = "https://hirehi.ru/api/search/jobs?limit=50"
page = requests.get(url)

In [3]:
page

<Response [200]>

In [4]:
print(page.json())

{'filter_counts': {'country': {}, 'direct_contact': 4669, 'english': 2451, 'format': {'гибрид': 1692, 'офис': 1497, 'удалённо': 4571, 'удалённо по РФ': 994}, 'hirehi': 98, 'level': {'head': 63, 'intern': 129, 'junior': 468, 'lead': 768, 'middle': 4235, 'senior': 3089, 'не указан': 1}, 'region': {}, 'salary': {'151–250k': 2886, '250k+': 4073, '81–150k': 990, 'до 80k': 781}}, 'has_more': True, 'initial_filter_counts': {'country': {}, 'direct_contact': 4669, 'english': 2451, 'format': {'гибрид': 1692, 'офис': 1497, 'удалённо': 4571, 'удалённо по РФ': 994}, 'hirehi': 98, 'level': {'head': 63, 'intern': 129, 'junior': 468, 'lead': 768, 'middle': 4235, 'senior': 3089, 'не указан': 1}, 'region': {}, 'salary': {'151–250k': 2886, '250k+': 4073, '81–150k': 990, 'до 80k': 781}}, 'jobs': [{'category': 'qa', 'company': 'NDA', 'company_icon': '', 'created_at': '2026-04-08T13:43:31Z', 'format': 'удалённо по РФ', 'id': 36700, 'is_from_recruiter': False, 'is_premium': False, 'level': 'senior', 'salary'

In [5]:
a = [page.json()["limit"], page.json()["page"], page.json()["total_count"]]
a

[50, 1, 8754]

In [6]:
import math

pageamount = math.ceil(a[2] / a[0])
pageamount

176

In [7]:
data = page.json()
jobs = data["jobs"]
first = jobs[0]
print(first.keys())

dict_keys(['category', 'company', 'company_icon', 'created_at', 'format', 'id', 'is_from_recruiter', 'is_premium', 'level', 'salary', 'salary_display', 'title'])


In [8]:
def get_page(p):
    url = f"https://hirehi.ru/api/search/jobs?page={p}&limit=50"
    response = requests.get(url)
    jobs = response.json()["jobs"]
    clean_data = []
    for i in jobs:
        cleanvac = {
            "id": i.get("id"),
            "title": i.get("title"),
            "category": i.get("category"),
            "company": i.get("company"),
            "format": i.get("format"),
            "level": i.get("level"),
            "salary": i.get("salary"),
        }
        clean_data.append(cleanvac)
    return clean_data

In [9]:
from tqdm.notebook import tqdm

In [10]:
infa = []
for p in tqdm(range(1, pageamount + 1)):
    try:
        infa.extend(get_page(p))
        sleep(3)
    except:
        print(p)

  0%|          | 0/176 [00:00<?, ?it/s]

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176


In [12]:
import pandas as pd

df = pd.DataFrame(infa)
print(df.shape)
df.head(10)

(8754, 7)


,id,title,category,company,format,level,salary
0,36700,quality assurance (manual),qa,NDA,удалённо по РФ,senior,от 240 000 ₽
1,36699,iOS разработчик,development,Экосистема тенниса,удалённо,senior,от 350 000 ₽
2,36697,platform engineer,devops,NDA,удалённо,senior,~ 372 000 ₽
3,36696,growth marketing manager,marketing,NDA,удалённо,middle,~ 223 000 ₽
4,36695,Бизнес аналитик,analytics,ВТБ,гибрид Москва,middle,~ 210 000 ₽
5,36694,DevOps инженер,devops,Сбер,гибрид Москва,senior,~ 345 000 ₽
6,36693,Python Developer,development,Сбер,гибрид Н.Новгород,middle,от 90 000 ₽
7,36692,c++ developer,development,Level 26 Games,удалённо,senior,от 320 000 ₽
8,36691,Аналитик данных,analytics,Магнит,гибрид Москва,intern,~ 65 000 ₽
9,36690,Системный аналитик,analytics,МТС,удалённо по РФ,senior,~ 282 000 ₽


Далее соберем ссылки:</br>
.NET Developer</br>
https://hirehi.ru/development/net-developer-36177</br>
графический дизайнер</br>
https://hirehi.ru/design/graficheskii-dizainer-36174</br>
Сылки выглядят таким образом , то есть https://hirehi.ru/{category}/{title}-{id}
</br>Все title приведены к нижнему регистру, текст на русском транслитерирован, все символы, кроме букв заменены на "-", крайние дефисы удалены, для этого можно использовать модуль **transliterate**.
Создадим дополнительные колонки для транслитерации русского текста в колонках **title** и **category**.

Если текст состоит из слов только на английском, то выводится none, значит текст, который написан на английском оставляем как есть. </br> Такжде слэши удаляются и не заменяются на дефисы, как должно быть с карточкой UX/UI дизайнер https://hirehi.ru/design/ux-ui-dizainer-36159

In [13]:
df["title_trans"] = df["title"]
df["id_trans"] = df["id"]
df.head(10)

,id,title,category,company,format,level,salary,title_trans,id_trans
0,36700,quality assurance (manual),qa,NDA,удалённо по РФ,senior,от 240 000 ₽,quality assurance (manual),36700
1,36699,iOS разработчик,development,Экосистема тенниса,удалённо,senior,от 350 000 ₽,iOS разработчик,36699
2,36697,platform engineer,devops,NDA,удалённо,senior,~ 372 000 ₽,platform engineer,36697
3,36696,growth marketing manager,marketing,NDA,удалённо,middle,~ 223 000 ₽,growth marketing manager,36696
4,36695,Бизнес аналитик,analytics,ВТБ,гибрид Москва,middle,~ 210 000 ₽,Бизнес аналитик,36695
5,36694,DevOps инженер,devops,Сбер,гибрид Москва,senior,~ 345 000 ₽,DevOps инженер,36694
6,36693,Python Developer,development,Сбер,гибрид Н.Новгород,middle,от 90 000 ₽,Python Developer,36693
7,36692,c++ developer,development,Level 26 Games,удалённо,senior,от 320 000 ₽,c++ developer,36692
8,36691,Аналитик данных,analytics,Магнит,гибрид Москва,intern,~ 65 000 ₽,Аналитик данных,36691
9,36690,Системный аналитик,analytics,МТС,удалённо по РФ,senior,~ 282 000 ₽,Системный аналитик,36690


In [14]:
df["title_trans"] = df["title_trans"].astype(str).str.lower()
df["id_trans"] = df["id_trans"].astype(str)
df.head(10)

,id,title,category,company,format,level,salary,title_trans,id_trans
0,36700,quality assurance (manual),qa,NDA,удалённо по РФ,senior,от 240 000 ₽,quality assurance (manual),36700
1,36699,iOS разработчик,development,Экосистема тенниса,удалённо,senior,от 350 000 ₽,ios разработчик,36699
2,36697,platform engineer,devops,NDA,удалённо,senior,~ 372 000 ₽,platform engineer,36697
3,36696,growth marketing manager,marketing,NDA,удалённо,middle,~ 223 000 ₽,growth marketing manager,36696
4,36695,Бизнес аналитик,analytics,ВТБ,гибрид Москва,middle,~ 210 000 ₽,бизнес аналитик,36695
5,36694,DevOps инженер,devops,Сбер,гибрид Москва,senior,~ 345 000 ₽,devops инженер,36694
6,36693,Python Developer,development,Сбер,гибрид Н.Новгород,middle,от 90 000 ₽,python developer,36693
7,36692,c++ developer,development,Level 26 Games,удалённо,senior,от 320 000 ₽,c++ developer,36692
8,36691,Аналитик данных,analytics,Магнит,гибрид Москва,intern,~ 65 000 ₽,аналитик данных,36691
9,36690,Системный аналитик,analytics,МТС,удалённо по РФ,senior,~ 282 000 ₽,системный аналитик,36690


In [15]:
df["title_trans"] = (
    df["title_trans"]
    .str.replace(r"[^\w]", "-", regex=True)
    .str.replace(r"_", "-", regex=True)
)
df

,id,title,category,company,format,level,salary,title_trans,id_trans
0,36700,quality assurance (manual),qa,NDA,удалённо по РФ,senior,от 240 000 ₽,quality-assurance--manual-,36700
1,36699,iOS разработчик,development,Экосистема тенниса,удалённо,senior,от 350 000 ₽,ios-разработчик,36699
2,36697,platform engineer,devops,NDA,удалённо,senior,~ 372 000 ₽,platform-engineer,36697
3,36696,growth marketing manager,marketing,NDA,удалённо,middle,~ 223 000 ₽,growth-marketing-manager,36696
4,36695,Бизнес аналитик,analytics,ВТБ,гибрид Москва,middle,~ 210 000 ₽,бизнес-аналитик,36695
...,...,...,...,...,...,...,...,...,...
8749,24856,ML Engineer,development,EvApps,удалённо,middle,от 180 до 220 ₽,ml-engineer,24856
8750,24838,Разработчик PHP,development,Системы документооборота,офис Казань,senior,от 150 000 ₽,разработчик-php,24838
8751,24836,Проектный менеджер IT,management,Системы документооборота,офис Казан,junior,None,проектный-менеджер-it,24836
8752,24787,Системный Аналитик,analytics,iStaff-IT,удалённо по РФ,senior,от 200 000 до 285 000 ₽,системный-аналитик,24787


нужно удалить повторяющиеся дефисы и дефисы в начале/конце

In [16]:
df["title_trans"] = df["title_trans"].str.replace(r"-+", "-", regex=True)
df

,id,title,category,company,format,level,salary,title_trans,id_trans
0,36700,quality assurance (manual),qa,NDA,удалённо по РФ,senior,от 240 000 ₽,quality-assurance-manual-,36700
1,36699,iOS разработчик,development,Экосистема тенниса,удалённо,senior,от 350 000 ₽,ios-разработчик,36699
2,36697,platform engineer,devops,NDA,удалённо,senior,~ 372 000 ₽,platform-engineer,36697
3,36696,growth marketing manager,marketing,NDA,удалённо,middle,~ 223 000 ₽,growth-marketing-manager,36696
4,36695,Бизнес аналитик,analytics,ВТБ,гибрид Москва,middle,~ 210 000 ₽,бизнес-аналитик,36695
...,...,...,...,...,...,...,...,...,...
8749,24856,ML Engineer,development,EvApps,удалённо,middle,от 180 до 220 ₽,ml-engineer,24856
8750,24838,Разработчик PHP,development,Системы документооборота,офис Казань,senior,от 150 000 ₽,разработчик-php,24838
8751,24836,Проектный менеджер IT,management,Системы документооборота,офис Казан,junior,None,проектный-менеджер-it,24836
8752,24787,Системный Аналитик,analytics,iStaff-IT,удалённо по РФ,senior,от 200 000 до 285 000 ₽,системный-аналитик,24787


In [17]:
df["title_trans"] = df["title_trans"].str.strip("-")
df

,id,title,category,company,format,level,salary,title_trans,id_trans
0,36700,quality assurance (manual),qa,NDA,удалённо по РФ,senior,от 240 000 ₽,quality-assurance-manual,36700
1,36699,iOS разработчик,development,Экосистема тенниса,удалённо,senior,от 350 000 ₽,ios-разработчик,36699
2,36697,platform engineer,devops,NDA,удалённо,senior,~ 372 000 ₽,platform-engineer,36697
3,36696,growth marketing manager,marketing,NDA,удалённо,middle,~ 223 000 ₽,growth-marketing-manager,36696
4,36695,Бизнес аналитик,analytics,ВТБ,гибрид Москва,middle,~ 210 000 ₽,бизнес-аналитик,36695
...,...,...,...,...,...,...,...,...,...
8749,24856,ML Engineer,development,EvApps,удалённо,middle,от 180 до 220 ₽,ml-engineer,24856
8750,24838,Разработчик PHP,development,Системы документооборота,офис Казань,senior,от 150 000 ₽,разработчик-php,24838
8751,24836,Проектный менеджер IT,management,Системы документооборота,офис Казан,junior,None,проектный-менеджер-it,24836
8752,24787,Системный Аналитик,analytics,iStaff-IT,удалённо по РФ,senior,от 200 000 до 285 000 ₽,системный-аналитик,24787


напишем функцию, которая будет транслитерировать русский текст, а английский оставлять как есть


In [18]:
def makelink1(i):
    letters = {
        "а": "a",
        "б": "b",
        "в": "v",
        "г": "g",
        "д": "d",
        "е": "e",
        "ё": "e",
        "ж": "zh",
        "з": "z",
        "и": "i",
        "й": "i",
        "к": "k",
        "л": "l",
        "м": "m",
        "н": "n",
        "о": "o",
        "п": "p",
        "р": "r",
        "с": "s",
        "т": "t",
        "у": "u",
        "ф": "f",
        "х": "h",
        "ц": "ts",
        "ч": "ch",
        "ш": "sh",
        "щ": "sch",
        "ъ": "",
        "ы": "y",
        "ь": "",
        "э": "e",
        "ю": "yu",
        "я": "ya",
    }
    ready = ""
    for s in i:
        if s in letters:
            ready += letters[s]
        else:
            ready += s
    return ready

In [19]:
df["title_trans"] = df["title_trans"].apply(makelink1)
df

,id,title,category,company,format,level,salary,title_trans,id_trans
0,36700,quality assurance (manual),qa,NDA,удалённо по РФ,senior,от 240 000 ₽,quality-assurance-manual,36700
1,36699,iOS разработчик,development,Экосистема тенниса,удалённо,senior,от 350 000 ₽,ios-razrabotchik,36699
2,36697,platform engineer,devops,NDA,удалённо,senior,~ 372 000 ₽,platform-engineer,36697
3,36696,growth marketing manager,marketing,NDA,удалённо,middle,~ 223 000 ₽,growth-marketing-manager,36696
4,36695,Бизнес аналитик,analytics,ВТБ,гибрид Москва,middle,~ 210 000 ₽,biznes-analitik,36695
...,...,...,...,...,...,...,...,...,...
8749,24856,ML Engineer,development,EvApps,удалённо,middle,от 180 до 220 ₽,ml-engineer,24856
8750,24838,Разработчик PHP,development,Системы документооборота,офис Казань,senior,от 150 000 ₽,razrabotchik-php,24838
8751,24836,Проектный менеджер IT,management,Системы документооборота,офис Казан,junior,None,proektnyi-menedzher-it,24836
8752,24787,Системный Аналитик,analytics,iStaff-IT,удалённо по РФ,senior,от 200 000 до 285 000 ₽,sistemnyi-analitik,24787


Теперь соберем ссылку в том виде, в котором она должна быть <br>
https://hirehi.ru/{category}/{title}-{id}


In [23]:
df["link"] = (
    "https://hirehi.ru/"
    + df["category"]
    + "/"
    + df["title_trans"]
    + "-"
    + df["id_trans"]
)
df

,id,title,category,company,format,level,salary,title_trans,id_trans,link
0,36700,quality assurance (manual),qa,NDA,удалённо по РФ,senior,от 240 000 ₽,quality-assurance-manual,36700,https://hirehi.ru/qa/quality-assurance-manual-...
1,36699,iOS разработчик,development,Экосистема тенниса,удалённо,senior,от 350 000 ₽,ios-razrabotchik,36699,https://hirehi.ru/development/ios-razrabotchik...
2,36697,platform engineer,devops,NDA,удалённо,senior,~ 372 000 ₽,platform-engineer,36697,https://hirehi.ru/devops/platform-engineer-36697
3,36696,growth marketing manager,marketing,NDA,удалённо,middle,~ 223 000 ₽,growth-marketing-manager,36696,https://hirehi.ru/marketing/growth-marketing-m...
4,36695,Бизнес аналитик,analytics,ВТБ,гибрид Москва,middle,~ 210 000 ₽,biznes-analitik,36695,https://hirehi.ru/analytics/biznes-analitik-36695
...,...,...,...,...,...,...,...,...,...,...
8749,24856,ML Engineer,development,EvApps,удалённо,middle,от 180 до 220 ₽,ml-engineer,24856,https://hirehi.ru/development/ml-engineer-24856
8750,24838,Разработчик PHP,development,Системы документооборота,офис Казань,senior,от 150 000 ₽,razrabotchik-php,24838,https://hirehi.ru/development/razrabotchik-php...
8751,24836,Проектный менеджер IT,management,Системы документооборота,офис Казан,junior,None,proektnyi-menedzher-it,24836,https://hirehi.ru/management/proektnyi-menedzh...
8752,24787,Системный Аналитик,analytics,iStaff-IT,удалённо по РФ,senior,от 200 000 до 285 000 ₽,sistemnyi-analitik,24787,https://hirehi.ru/analytics/sistemnyi-analitik...


мы получили ссылки, но можно заметить такую проблему, что transliterate заменяет буквы не так "по-русски", в наших ссылках пишется **ux-ui-dizajner**, когда в оригинальной ссылке используется **ux-ui-dizainer**
значит библиотека не сможет "правильно" транслитерировать наш текст. Поэтому напишем словарь, с помощью которого можно будет просто заменять русскую букву на английскую, без дополнительных правил библиотеки, когда две гласные стоят рядом и происходит удвоение.


In [24]:
df.drop(columns=["title_trans", "id_trans"], inplace=True)
df

,id,title,category,company,format,level,salary,link
0,36700,quality assurance (manual),qa,NDA,удалённо по РФ,senior,от 240 000 ₽,https://hirehi.ru/qa/quality-assurance-manual-...
1,36699,iOS разработчик,development,Экосистема тенниса,удалённо,senior,от 350 000 ₽,https://hirehi.ru/development/ios-razrabotchik...
2,36697,platform engineer,devops,NDA,удалённо,senior,~ 372 000 ₽,https://hirehi.ru/devops/platform-engineer-36697
3,36696,growth marketing manager,marketing,NDA,удалённо,middle,~ 223 000 ₽,https://hirehi.ru/marketing/growth-marketing-m...
4,36695,Бизнес аналитик,analytics,ВТБ,гибрид Москва,middle,~ 210 000 ₽,https://hirehi.ru/analytics/biznes-analitik-36695
...,...,...,...,...,...,...,...,...
8749,24856,ML Engineer,development,EvApps,удалённо,middle,от 180 до 220 ₽,https://hirehi.ru/development/ml-engineer-24856
8750,24838,Разработчик PHP,development,Системы документооборота,офис Казань,senior,от 150 000 ₽,https://hirehi.ru/development/razrabotchik-php...
8751,24836,Проектный менеджер IT,management,Системы документооборота,офис Казан,junior,None,https://hirehi.ru/management/proektnyi-menedzh...
8752,24787,Системный Аналитик,analytics,iStaff-IT,удалённо по РФ,senior,от 200 000 до 285 000 ₽,https://hirehi.ru/analytics/sistemnyi-analitik...


In [25]:
df.to_csv("hirehi_parsing.csv", index=False)